### EVAL
Para realizar las predicciones de los tests y genere su rendimiento.

In [10]:
# Importaciones
from ultralytics import YOLO
import numpy as np
import pandas as pd
import glob
import os
import pickle
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, classification_report)
from sklearn.preprocessing import LabelBinarizer

### Localización de los modelos entrenados

In [11]:
# Ruta del conjunto de datos de prueba
PATH_TEST_DATASET = "dataset_isic/test"
# Ruta donde buscar los pesos entrenados
SEARCH_PATTERN = "runs/classify/*/weights/best.pt"

# Rutas donde se guardarán los resultados de la evaluación
OUTPUT_CSV = "evaluacion_metricas.csv"
OUTPUT_PKL = "datos_para_roc.pkl"

### Funciones auxiliares

In [12]:
def calcular_metricas_avanzadas(y_true, y_pred, y_probs, class_names):
    """
    Calcula métricas detalladas: Especificidad, FNR, FPR y AUC.
    Maneja el enfoque One-vs-Rest para problemas multiclase.
    """
    # 1. Matriz de Confusión
    cm = confusion_matrix(y_true, y_pred)
    
    # Cálculos One-vs-Rest (para cada clase)
    FP = cm.sum(axis=0) - np.diag(cm)  
    FN = cm.sum(axis=1) - np.diag(cm)
    TP = np.diag(cm)
    TN = cm.sum() - (FP + FN + TP)

    # 2. Promediar métricas (Macro Average)
    
    # Especificidad = TN / (TN + FP)
    with np.errstate(divide='ignore', invalid='ignore'):
        spec_per_class = TN / (TN + FP)
        fnr_per_class = FN / (TP + FN) # Tasa Falsos Negativos
        fpr_per_class = FP / (FP + TN) # Tasa Falsos Positivos
    
    specificity = np.nanmean(spec_per_class)
    fnr = np.nanmean(fnr_per_class)
    fpr = np.nanmean(fpr_per_class)
    
    # 3. AUC (Area Under Curve) - Multiclase
    # Necesitamos binarizar las etiquetas reales para scikit-learn
    try:
        lb = LabelBinarizer()
        lb.fit(y_true)
        y_true_bin = lb.transform(y_true)
        
        # Si hay pocas clases en el test, aseguramos la forma correcta
        if y_true_bin.shape[1] != y_probs.shape[1]:
            # Caso borde: si el test no tiene todas las clases que el modelo conoce
            auc_score = 0.0 
        else:
            auc_score = roc_auc_score(y_true_bin, y_probs, multi_class='ovr', average='macro')
    except Exception as e:
        print(f"Warning: No se pudo calcular AUC ({e})")
        auc_score = 0.0
    
    print("\n--- Desglose por Clase ---")
    # Convertimos el diccionario {0: 'Melanoma', ...} a una lista ordenada de nombres
    nombres_ordenados = [class_names[i] for i in range(len(class_names))]
    print(classification_report(y_true, y_pred, target_names=nombres_ordenados))

    return specificity, fnr, fpr, auc_score

### Evaluar modelos y guardar métricas

In [13]:
imagenes_test = glob.glob(os.path.join(PATH_TEST_DATASET, "**", "*.jpg"), recursive=True)
rutas_modelos = glob.glob(SEARCH_PATTERN)
print(f"Modelos encontrados: {len(rutas_modelos)}")
    
resultados_tabla = []
datos_para_graficas = {} # Diccionario para guardar datos crudos para el siguiente script

for ruta in rutas_modelos:
    # Extraer nombre limpio del modelo (ej: yolov8n)
    nombre_modelo = ruta.split(os.sep)[-3].replace("train_", "")
    print(f"\n--- Evaluando modelo: {nombre_modelo} ---")
        
    # Cargar modelo
    try:
        model = YOLO(ruta)
    except Exception as e:
        print(f"Error cargando {ruta}: {e}")
        continue
            
    # Realizar predicción
    # stream=True es vital para no saturar la RAM con muchas imágenes
    results = model.predict(source=imagenes_test, stream=True, verbose=False)
        
    y_true = []
    y_pred = []
    y_probs = []
        
    print("Procesando imágenes...")
        
    # Iterar sobre los resultados de la predicción
    for r in results:
        # --- Extracción de la etiqueta real basada en la ruta del archivo ---
        path_parts = r.path.split(os.sep)
        true_class_name = path_parts[-2] 
            
        # Mapear nombre de carpeta a índice numérico del modelo
        # model.names es un dict {0: 'Melanoma', 1: 'Nevus'...}
        name_to_idx = {v: k for k, v in model.names.items()}
        true_idx = name_to_idx.get(true_class_name)
            
        if true_idx is not None:
            y_true.append(true_idx)
                
            # Predicción (índice de la clase con mayor probabilidad)
            top1_idx = r.probs.top1
            y_pred.append(top1_idx)
                
            # Vector de probabilidades (necesario para ROC)
            y_probs.append(r.probs.data.cpu().numpy())

    # Convertir a arrays de numpy para cálculo eficiente
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_probs = np.array(y_probs)

    if len(y_true) == 0:
        print("Error: No se encontraron etiquetas válidas. Revisa PATH_TEST_DATASET.")
        continue

    # --- CÁLCULO DE MÉTRICAS STANDARD (Scikit-Learn) ---
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0) # Sensibilidad
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        
    # --- CÁLCULO DE MÉTRICAS AVANZADAS ---
    spec, fnr, fpr, auc = calcular_metricas_avanzadas(y_true, y_pred, y_probs, model.names)

    print(f"Resultados -> Acc: {acc:.4f} | F1: {f1:.4f} | Sens: {recall:.4f} | Spec: {spec:.4f} | AUC: {auc:.4f}")
        
    # Agregar a la lista de resultados
    resultados_tabla.append({
        "Modelo": nombre_modelo,
        "Accuracy": round(acc, 4),
        "Precision": round(precision, 4),
        "Recall (Sensibilidad)": round(recall, 4),
        "F1-Score": round(f1, 4),
        "Especificidad": round(spec, 4),
        "FNR": round(fnr, 4),
        "FPR": round(fpr, 4),
        "AUC": round(auc, 4),
        "Ruta Pesos": ruta
    })
        
    # Guardar datos crudos para graficar ROC posteriormente
    datos_para_graficas[nombre_modelo] = {
        "y_true": y_true,
        "y_probs": y_probs,
        "class_names": model.names
    }

    if resultados_tabla:
        # 1. Guardar CSV
        df = pd.DataFrame(resultados_tabla)
        df.to_csv(OUTPUT_CSV, index=False)
        print(f"\n[OK] Tabla de métricas guardada en: {OUTPUT_CSV}")
        print(df.to_string(index=False))
        
        # 2. Guardar Pickle (para usar en results.ipynb para las gráficas)
        with open(OUTPUT_PKL, 'wb') as f:
            pickle.dump(datos_para_graficas, f)
        print(f"[OK] Datos para gráficas ROC guardados en: {OUTPUT_PKL}")
    else:
        print("\n[ERROR] No se generaron resultados. Verifica las rutas.")

Modelos encontrados: 3

--- Evaluando modelo: yolov8m-cls ---
Procesando imágenes...

--- Desglose por Clase ---
              precision    recall  f1-score   support

          AK       0.70      0.70      0.70        10
         BCC       0.46      0.60      0.52        10
         BKL       0.83      0.50      0.62        10
          DF       0.80      0.80      0.80        10
         MEL       1.00      0.60      0.75        10
          NV       0.62      0.80      0.70        10
         SCC       0.55      0.60      0.57        10
        VASC       0.91      1.00      0.95        10

    accuracy                           0.70        80
   macro avg       0.73      0.70      0.70        80
weighted avg       0.73      0.70      0.70        80

Resultados -> Acc: 0.7000 | F1: 0.7020 | Sens: 0.7000 | Spec: 0.9571 | AUC: 0.9291

[OK] Tabla de métricas guardada en: evaluacion_metricas.csv
     Modelo  Accuracy  Precision  Recall (Sensibilidad)  F1-Score  Especificidad  FNR    FPR